### <font color=Blue><b>This Jupyter notebook demonstrates: Private data ingestion and use of vector databases with LangChain

### <font color=Blue> Disable Warnings

In [1]:
import warnings
warnings.filterwarnings("ignore")
from getpass import getpass
import os

### <font color=Blue>**Create LLM Instance**

In [3]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content


'Hello! How can I help you today?'

### <font color=Blue>**Working with text files**

Use below code snippet to download text file if not available

In [3]:
#import requests
#url="https://github.com/labmlai/annotated_deep_learning_paper_implementations/blob/master/requirements.txt"
#url = "https://github.com/hwchase17/chat-your-data/blob/master/state_of_the_union.txt"

#result=requests.get(url)
#with open("requirements.txt", "w") as file:
 #file.write(result.text) 


### <font color=Blue> Import the necessary Packages and Read the File

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
#from langchain.text_splitter import CharacterTextSplitter

from langchain_community.vectorstores import FAISS

import textwrap

loader=TextLoader(r"state_of_the_union.txt")
documents = loader.load()


In [9]:
documents

[Document(metadata={'source': 'state_of_the_union.txt'}, page_content='"Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  ","","Last year COVID-19 kept us apart. This year we are finally together again. ","","Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. ","","With a duty to one another to the American people to the Constitution. ","","And with an unwavering resolve that freedom will always triumph over tyranny. ","","Six days ago, Russia s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. ","","He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. ","","He met the Ukrainian people. ","","From President Zelenskyy to every Ukrainian, their fearlessness, their courag

### <font color=Blue>  Download the Embedding Model and Use it Locally

In [10]:
from langchain_aws import BedrockEmbeddings
embeddings=BedrockEmbeddings(model_id='cohere.embed-english-v3', #amazon.titan-embed-text-v1
                        aws_access_key_id='',
                        aws_secret_access_key='',
                       region_name='us-east-1')


### <font color=Blue> Split the Text and Create VDB

In [11]:
# Initialize a text splitter to break large text into smaller chunks
# - chunk_size: Maximum number of characters per chunk
# - chunk_overlap: Number of overlapping characters between chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter


#  Use recursive chunking for more natural splits (prefers paragraph → sentence → word)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,         # Max characters per chunk
    chunk_overlap=100,       # Overlap between chunks
    separators=["\n\n", "\n", ".", " ", ""]  # Tries to split first at paragraph, then sentence, etc.
)

# Split the input documents into smaller chunks
docs = text_splitter.split_documents(documents)

docs

[Document(metadata={'source': 'state_of_the_union.txt'}, page_content='"Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  ","","Last year COVID-19 kept us apart. This year we are finally together again. ","","Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. ","","With a duty to one another to the American people to the Constitution. ","","And with an unwavering resolve that freedom will always triumph over tyranny. ","","Six days ago, Russia s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. ","","He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. ","","He met the Ukrainian people. ","","From President Zelenskyy to every Ukrainian, their fearlessness, their courag

In [12]:
# Create FAISS vector store from the chunks
db = FAISS.from_documents(docs, embeddings)


In [13]:
#optional - saves index to disk
    
db.save_local("faiss_store")  

In [15]:
# Load the FAISS vector store - if already stored

from langchain_community.vectorstores import FAISS
faiss_db = FAISS.load_local("faiss_store", embeddings=embeddings,allow_dangerous_deserialization=True)

results = faiss_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [16]:
results.invoke("What was the action taken against Russian central bank?")

[Document(id='d0cdb49e-4b69-4661-bb43-5a033d4e53d0', metadata={'source': 'state_of_the_union.txt'}, page_content='.   ","","And now that he has acted the free world is holding him accountable. ","","Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. ","","We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. ","","Together with our allies  we are right now enforcing powerful economic sanctions. ","","We are cutting off Russia s largest banks from the international financial system.  ","","Preventing Russia s central bank from defending the Russian Ruble making Putin s $630 Billion  war fund  worthless.   ","","We are choking off Russia s access to technology that will sap its economic strength and weaken its military for years to come.  ","","Tonight I say

##### To know the vector stores available in Langchain pls visit the website : https://python.langchain.com/docs/concepts/vectorstores/


### <font color=Blue>Perform Similarity Search for the Given Query
The returned distance score is based on cosine distance; therefore, a lower score indicates a better match.

In [17]:
#Run a similarity search
query1 = "What was the action taken against Russian central bank?"
results = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [18]:
chunks=results.invoke(query1)

In [19]:
#Display retrieved chunks
for i, doc in enumerate(chunks):
    print(f"\n🔹 Chunk {i+1}:\n{doc}")


🔹 Chunk 1:
page_content='.   ","","And now that he has acted the free world is holding him accountable. ","","Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. ","","We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. ","","Together with our allies  we are right now enforcing powerful economic sanctions. ","","We are cutting off Russia s largest banks from the international financial system.  ","","Preventing Russia s central bank from defending the Russian Ruble making Putin s $630 Billion  war fund  worthless.   ","","We are choking off Russia s access to technology that will sap its economic strength and weaken its military for years to come.  ","","Tonight I say to the Russian oligarchs and corrupt leaders who have bilked billions of dollars off th

In [20]:
template = """Question: {question} and retrieved docs:{input_documents} Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question",'input_documents'])

In [21]:
prompt

PromptTemplate(input_variables=['input_documents', 'question'], input_types={}, partial_variables={}, template="Question: {question} and retrieved docs:{input_documents} Answer: Let's think step by step.")

In [23]:
from langchain_core.output_parsers import StrOutputParser
llm_chain = prompt | llm | StrOutputParser()

In [24]:
print(llm_chain.invoke({'input_documents':chunks,'question':query1}))

The US and its allies prevented Russia's central bank from defending the Russian Ruble, making Putin's $630 Billion war fund worthless.


In [25]:
docs_and_scores = db.similarity_search_with_score(query1,k=2)

In [26]:
docs_and_scores

[(Document(id='d0cdb49e-4b69-4661-bb43-5a033d4e53d0', metadata={'source': 'state_of_the_union.txt'}, page_content='.   ","","And now that he has acted the free world is holding him accountable. ","","Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. ","","We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. ","","Together with our allies  we are right now enforcing powerful economic sanctions. ","","We are cutting off Russia s largest banks from the international financial system.  ","","Preventing Russia s central bank from defending the Russian Ruble making Putin s $630 Billion  war fund  worthless.   ","","We are choking off Russia s access to technology that will sap its economic strength and weaken its military for years to come.  ","","Tonight I sa

# <font color=Blue> **Working with PDF files**

<font color=Blue> 
This implementation uses Chroma DB as the Vector Database (VDB)
for storing and retrieving document embeddings efficiently.
Chroma provides persistent storage and fast similarity search capabilities.


For more details on Chroma with LangChain follow the link below:

<font color=Blue>**https://python.langchain.com/docs/integrations/vectorstores/chroma/**

In [3]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content


'Hello! How can I help you today?'

In [4]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

# load document
loader = PyPDFLoader(r"IntroToCloud.pdf")
documents = loader.load()

#  Use recursive chunking for more natural splits (prefers paragraph → sentence → word)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,         # Max characters per chunk
    chunk_overlap=50,       # Overlap between chunks
    separators=["\n\n", "\n", ".", " ", ""]  # Tries to split first at paragraph, then sentence, etc.
)

texts = text_splitter.split_documents(documents)

In [7]:
documents[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-30T12:13:33+05:30', 'author': 'Sushobhit Kumar', 'moddate': '2023-10-30T12:13:33+05:30', 'source': 'IntroToCloud.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='What is KMP? \nKotlin MultiPlatform(KMP) is a feature released in 2017 with Kotlin 1.2. This feature allows developers to \nwrite shared code that can be used across multiple platforms. It enables code reuse between different \nplatforms, such as Android, iOS, web, desktop, and others. With KMP , you can write common business \nlogic, data models, algorithms, and other core components in Kotlin and share them across platforms. \nThis reduces the need for writing separate codebases for each platform, saving development time and \neffort. \n \nFeatures of KMM:- \n1. Shared code:- KMM allows developers to share business logic, data models, networking code, \nand even UI-r

In [5]:
texts[0]

Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-30T12:13:33+05:30', 'author': 'Sushobhit Kumar', 'moddate': '2023-10-30T12:13:33+05:30', 'source': 'IntroToCloud.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='What is KMP? \nKotlin MultiPlatform(KMP) is a feature released in 2017 with Kotlin 1.2. This feature allows developers to \nwrite shared code that can be used across multiple platforms. It enables code reuse between different \nplatforms, such as Android, iOS, web, desktop, and others. With KMP , you can write common business \nlogic, data models, algorithms, and other core components in Kotlin and share them across platforms.')

In [6]:
from langchain_aws import BedrockEmbeddings
embeddings=BedrockEmbeddings(model_id="amazon.titan-embed-text-v1",
                        aws_access_key_id='',
                        aws_secret_access_key='',
                       region_name='us-east-1')

<font color=red>More vector stores can be explored by practitioners following below link.

### <font color=Blue> https://python.langchain.com/docs/integrations/vectorstores/

In [8]:
# Import Chroma vector store from LangChain

from langchain_community.vectorstores import Chroma

# Import ChromaDB configuration (optional, useful for advanced setup or customization)
import chromadb.config

# Create the vector store using Chroma from a list of documents and their embeddings
# This will serve as the index for similarity search and retrieval
db = Chroma.from_documents(texts, embeddings)

In [14]:
db.similarity_search("What is cloud computing?",k=2)

[Document(metadata={'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'page': 1, 'moddate': '2023-10-30T12:13:33+05:30', 'source': 'IntroToCloud.pdf', 'creationdate': '2023-10-30T12:13:33+05:30', 'total_pages': 3, 'page_label': '2', 'author': 'Sushobhit Kumar'}, page_content='There are three different backend compilers: one for each of the Java virtual machine (JVM), JavaScript \n(JS) and native targets. \n \nWhat is cloud computing? \nCloud computing means that all the computing hardware and software resources that you need to \nprocess your tasks are provided for you, "as a service" over the internet, by a vendor instead of you \nowning and maintaining them. \nIt is the duty of the vendor, the Cloud Service Provider (CSP), to develop, own and maintain these'),
 Document(metadata={'moddate': '2023-10-30T12:13:33+05:30', 'total_pages': 3, 'author': 'Sushobhit Kumar', 'page_label': '2', 'creationdate': '2023-10-30T12:13:33+05:30', 'source':

In [9]:
# from langchain_chroma import Chroma

# # Define a folder to persist the database
# persist_dir = "chroma_db2"

# # Create and persist the vector store
# db = Chroma.from_documents(documents, embeddings, persist_directory=persist_dir)
# # db.persist()  # Save the DB to disk


In [10]:
# from langchain_chroma import Chroma

# # # Load the persisted vector store
# db = Chroma(persist_directory="chroma_db2", embedding_function=embeddings)

# # # Now you can use db.similarity_search(), etc.


In [11]:

# Import RetrievalQA chain from LangChain
from langchain_classic.chains import RetrievalQA

# Convert the Chroma vector store into a retriever interface
# search_type="similarity" performs similarity-based retrieval
# search_kwargs={"k": 2} means it will return the top 2 most similar documents

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# Create a RetrievalQA chain using the retriever and a language model (llm)
# chain_type="stuff" means it will concatenate retrieved documents before passing to the LLM
# return_source_documents=True allows you to see which documents were used to generate the answer

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    verbose=True,
    retriever=retriever,
    return_source_documents=True
)


In [41]:
#retriever.invoke("Explain cloud computing ?")

In [12]:
query = "What is cloud computing ?"

result = qa.invoke(query,verbose=True)
print(result['result'])



> Entering new RetrievalQA chain...

> Finished chain.
Cloud computing is a model where a vendor, known as a Cloud Service Provider (CSP), provides hardware and software resources to customers over the internet, "as a service". This means that instead of owning and maintaining their own computing infrastructure, customers can access and use these resources on-demand, paying only for what they use. The CSP is responsible for developing, owning, and maintaining the resources, which can include processing power, storage, databases, and software applications.


In [42]:
query = "What is cloud computing ?"

result = qa.invoke(query,verbose=True)
print(result['result'])



> Entering new RetrievalQA chain...

> Finished chain.
Cloud computing is a model that allows users to access computing resources, such as hardware and software, over the internet, rather than owning and maintaining these resources themselves. The cloud service provider (CSP) is responsible for developing, owning, and maintaining these resources and making them available to consumers on a pay-as-you-go basis. This provides consumers with the ability to scale their computing resources up or down as needed, without having to invest in and manage their own infrastructure.


In [43]:
query = "What is Machine Learning?"

result = qa.invoke(query,verbose=True)
print(result['result'])



> Entering new RetrievalQA chain...

> Finished chain.
Sorry, I don't know the answer to that.


In [ ]:
## Original Text from PDF

Cloud computing means that all the computing hardware and software resources that you need to 
process your tasks are provided for you, "as a service" over the internet, by a vendor instead of you 
owning and maintaining them.
It is the duty of the vendor, the Cloud Service Provider (CSP), to develop, own and maintain these 
resources and make them accessible to the consumers over the internet.

FAISS and ChromaDB

| Feature             | **FAISS**                                   | **ChromaDB**                                     |
| ------------------- | ------------------------------------------- | ------------------------------------------------ |
| Backend             | C++ core (via Python bindings)              | Rust (wrapped in Python)                         |
| Speed / Performance | 🚀 Very fast for large-scale vector search  | ⚡ Fast enough, good for smaller/mid-scale setups |
| Index types         | Flat, IVF, HNSW, PQ, etc. (highly tunable)  | Flat only (as of now)                            |
| Metadata filtering  | ❌ Limited (external)                        | ✅ Native filtering (`where={"field": value}`)    |
| Persistence         | 🟡 Manual (`save_local`, `load_local`)      | ✅ Built-in (`persist()`, `persist_directory`)    |
| Document management | ❌ External (LangChain wraps this)           | ✅ Native (text + metadata + vector stored)       |
| Scalability         | ✅ Very high (used in billion-scale setups)  | 🟡 Moderate scale (on local or cloud edge)       |
| Query filtering     | ❌ No filtering (unless custom wrapper)      | ✅ Powerful metadata filtering                    |
| Integration         | ✅ LangChain, Haystack                       | ✅ LangChain, LlamaIndex                          |
| Use case type       | Best for **pure vector search**, large data | Best for **RAG apps**, with filtering & metadata |
| Ecosystem           | Broad (used in industry heavily)            | Growing rapidly with RAG focus                   |


| Retriever Type                                          | Embedding Needed? | Best Use Case                                          | Strengths                                                       | Weaknesses                             |
| ------------------------------------------------------- | ----------------- | ------------------------------------------------------ | --------------------------------------------------------------- | -------------------------------------- |
| **VectorStoreRetriever** (e.g. FAISS, Chroma, Pinecone) | ✅ Yes             | Unstructured documents, semantic search                | Finds semantically similar content, even if phrased differently | Needs good embeddings, harder to debug |
| **BM25Retriever**                                       | ❌ No              | Structured or short documents, keyword-heavy text      | Fast, interpretable, no embeddings needed                       | No understanding of semantics          |
| **HybridRetriever** (BM25 + Vectors)                    | ✅ Optional        | When both semantics and keyword matching are important | Combines strengths of both BM25 and vectors                     | Slightly complex to set up             |
| **Wikipedia / TavilyRetriever**                         | ❌ No              | External general knowledge                             | Always fresh, broad coverage                                    | Not good for custom/private data       |
| **SQL/Tabular Retriever**                               | ❌ No              | Relational data, tables                                | Accurate, queryable                                             | Not good for natural text              |
| **Custom Rule-based Retriever**                         | ❌ No              | Domain-specific tasks                                  | Highly tailored                                                 | Low generalizability                   |


### 🔢 **Choosing the Right Chunk Size in RAG**

The **chunk size** (in tokens or characters) is a critical hyperparameter in RAG pipelines. It determines how much text is included in each chunk passed to the vector store and eventually to the language model.

---

### ✅ **General Guidelines for Chunk Size**

| Use Case                                  | Recommended Chunk Size (tokens) | Notes                                                       |
| ----------------------------------------- | ------------------------------- | ----------------------------------------------------------- |
| **General text (e.g., blog, wiki)**       | 300–500 tokens                  | Balances semantic completeness with retrievability          |
| **Legal/academic documents**              | 500–1000 tokens                 | Chunks should preserve logical structure (e.g., paragraphs) |
| **Conversational data (e.g., chat logs)** | 100–300 tokens                  | Smaller chunks to preserve speaker turns                    |
| **Code documents**                        | 100–200 tokens                  | Code is sensitive to context and indentation                |
| **PDFs with irregular layout**            | 200–400 tokens                  | Consider paragraph-based or heuristic chunking              |

> 📌 **Rule of Thumb**: Use `chunk_overlap = 10–20%` of chunk size (e.g., 50–100 tokens overlap for a 500-token chunk).

---

### ⚠️ Things to Watch

* **Too small** chunks (e.g., <100 tokens): may lack context and lower retrieval precision.
* **Too large** chunks (e.g., >1000 tokens): may not fit in LLM context or dilute semantic focus.

---


Chunk size in a RAG system should be **task-dependent**, as the goal is to optimize retrieval relevance **and** LLM response quality. Here's a breakdown by **task type** with suggested chunk sizes and rationale:

---

### 🧠 **Chunk Size Recommendations by Task**

| **Task Type**                      | **Recommended Chunk Size** | **Reasoning**                                                              |
| ---------------------------------- | -------------------------- | -------------------------------------------------------------------------- |
| **Question Answering (QA)**        | 300–500 tokens             | Ensures answer context is complete without exceeding input limits          |
| **Summarization**                  | 500–1000 tokens            | Larger chunks retain more context for coherent summaries                   |
| **Fact Verification**              | 200–400 tokens             | Keeps facts isolated, precise, and easily retrievable                      |
| **Code Explanation/Generation**    | 100–200 tokens             | Preserves syntactic structure; avoids mixing multiple logic blocks         |
| **Legal/Medical Retrieval**        | 500–800 tokens             | Captures full sections or clauses necessary for semantic integrity         |
| **Semantic Search/Classification** | 200–300 tokens             | Smaller, cleaner chunks improve retrieval accuracy and relevance scoring   |
| **Multi-turn Dialogue Retrieval**  | 100–300 tokens             | Captures conversation turns; prevents retrieval of irrelevant past context |
| **Scientific Paper Q\&A**          | 400–700 tokens             | Preserves section-level context (e.g., method/results paragraphs)          |

---

### 🔁 **Chunk Overlap Recommendation by Task**

| **Task**           | **Overlap**   | **Why**                                      |
| ------------------ | ------------- | -------------------------------------------- |
| QA, Fact Retrieval | 50–100 tokens | Preserves sentence continuity across chunks  |
| Summarization      | 0–50 tokens   | Less needed; models summarize bigger context |
| Code Tasks         | 50–80 tokens  | Preserves code blocks/scope                  |
| Dialogue-based QA  | 30–50 tokens  | Maintains speaker coherence                  |

---


### 🧩 Tip

If using an **LLM with small context window** (e.g., 4k tokens):
→ prefer smaller chunks (≤ 500 tokens)

If using **larger models** (e.g., GPT-4-32k, Claude 200k):
→ can use chunk size up to 1000+ tokens, reducing retrieval noise.

---
